# Complete Audio ML Training Pipeline
**MagnaTagATune → CNN Classifier → Transformer Generator → SVDQuant → ONNX Export**

This notebook consolidates the full training pipeline with minimal, working code.

In [ ]:
# ============================================
# SECTION 1: ENVIRONMENT SETUP
# ============================================

!pip install -q torch torchaudio librosa numpy pandas matplotlib scikit-learn tqdm onnxscript
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================
# SECTION 2: CONFIGURATION
# ============================================

CONFIG = {
    'data_dir': '/content/magnatagatune',
    'output_dir': Path('./models'),
    'model_save_dir': Path('./models'),
    'sample_rate': 22050,
    'duration': 5,
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'batch_size': 32,
    'epochs': 10,
    'learning_rate': 1e-4,
    'target_tags': ['drums', 'bass', 'synth', 'electronic', 'dance', 'beat', 'percussion', 'deep', 'house'],
    'max_train_samples': 4000,
    'max_val_samples': 1000,
}

CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)
CONFIG['model_save_dir'].mkdir(parents=True, exist_ok=True)
print(f"Configuration loaded. Output dir: {CONFIG['output_dir']}")

In [ ]:
# ============================================
# SECTION 3: DATASET DOWNLOAD & PREPARATION
# ============================================

import os
import zipfile

DATASET_URL = "https://huggingface.co/datasets/confit/magnatagatune/resolve/main/mp3.zip"
DATASET_DIR = Path(CONFIG['data_dir'])
ZIP_PATH = DATASET_DIR / "mp3.zip"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Download dataset
if not ZIP_PATH.exists():
    print(f"Downloading dataset (~1.5GB)...")
    !wget -q --show-progress -O {ZIP_PATH} {DATASET_URL}
    print("Download complete!")
else:
    print("Dataset already downloaded.")

# Extract
mp3_files_exist = len(list(DATASET_DIR.rglob('*.mp3'))) > 0
if not mp3_files_exist:
    print("Extracting audio files...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Extraction complete!")

# Download annotations
!wget -q -nc -O {DATASET_DIR}/annotations_final.csv https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv 2>/dev/null || true

# Find all MP3 files
ALL_MP3_FILES = list(DATASET_DIR.rglob('*.mp3'))
print(f"\n✓ Dataset ready at {DATASET_DIR}")
print(f"✓ Found {len(ALL_MP3_FILES)} MP3 files")

In [ ]:
# ============================================
# SECTION 4: DATASET CLASS (with librosa)
# ============================================

class MagnaTagATuneDataset(Dataset):
    """MagnaTagATune dataset with librosa for reliable MP3 loading"""
    
    def __init__(self, config, split='train', transform=None):
        self.config = config
        self.transform = transform
        self.sr = config['sample_rate']
        self.duration = config['duration']
        self.target_length = self.sr * self.duration
        
        # Load annotations
        annotations_path = Path(config['data_dir']) / 'annotations_final.csv'
        print(f"Loading annotations from {annotations_path}")
        
        self.annotations = pd.read_csv(annotations_path, sep='\t')
        
        # Select Amapiano-relevant tags
        self.tag_columns = [col for col in config['target_tags'] if col in self.annotations.columns]
        print(f"Using {len(self.tag_columns)} tags: {self.tag_columns}")
        
        # Build file list with labels
        self.file_list = []
        self.labels = []
        
        mp3_dir = Path(config['data_dir'])
        all_mp3s = {f.name: f for f in mp3_dir.rglob('*.mp3')}
        print(f"Found {len(all_mp3s)} MP3 files")
        
        for idx, row in self.annotations.iterrows():
            mp3_name = Path(row['mp3_path']).name if 'mp3_path' in row else None
            if mp3_name and mp3_name in all_mp3s:
                self.file_list.append(all_mp3s[mp3_name])
                label = torch.tensor([row[tag] if tag in row else 0 for tag in self.tag_columns], dtype=torch.float32)
                self.labels.append(label)
        
        print(f"Matched {len(self.file_list)} files with annotations")
        
        # Split dataset
        total = len(self.file_list)
        train_end = int(total * 0.8)
        
        if split == 'train':
            self.file_list = self.file_list[:train_end]
            self.labels = self.labels[:train_end]
        else:
            self.file_list = self.file_list[train_end:]
            self.labels = self.labels[train_end:]
        
        # Limit for faster training
        max_samples = config.get('max_train_samples', 4000) if split == 'train' else config.get('max_val_samples', 1000)
        self.file_list = self.file_list[:max_samples]
        self.labels = self.labels[:max_samples]
        
        print(f"Dataset ({split}): {len(self.file_list)} files")
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        audio_path = self.file_list[idx]
        label = self.labels[idx]
        
        try:
            waveform, _ = librosa.load(str(audio_path), sr=self.sr, mono=True, duration=self.duration)
            waveform = torch.from_numpy(waveform).float()
        except Exception as e:
            waveform = torch.zeros(self.target_length)
        
        # Ensure correct length
        if len(waveform) < self.target_length:
            waveform = F.pad(waveform, (0, self.target_length - len(waveform)))
        elif len(waveform) > self.target_length:
            waveform = waveform[:self.target_length]
        
        # Normalize
        if waveform.abs().max() > 0:
            waveform = waveform / waveform.abs().max() * 0.95
        
        if self.transform:
            waveform = self.transform(waveform)
        
        return waveform, label

print("Dataset class defined.")

In [ ]:
# ============================================
# SECTION 5: DATA AUGMENTATION
# ============================================

class AudioAugmentation:
    """Audio augmentation for training robustness."""
    
    def __init__(self, p=0.5):
        self.p = p
    
    def add_noise(self, waveform, noise_level=0.005):
        noise = torch.randn_like(waveform) * noise_level
        return waveform + noise
    
    def random_gain(self, waveform, min_gain=0.8, max_gain=1.2):
        gain = torch.empty(1).uniform_(min_gain, max_gain)
        return waveform * gain
    
    def time_shift(self, waveform, max_shift=0.1):
        shift = int(waveform.shape[-1] * max_shift * torch.rand(1))
        return torch.roll(waveform, shift, dims=-1)
    
    def __call__(self, waveform):
        if torch.rand(1) < self.p:
            waveform = self.add_noise(waveform)
        if torch.rand(1) < self.p:
            waveform = self.random_gain(waveform)
        if torch.rand(1) < self.p:
            waveform = self.time_shift(waveform)
        return waveform

augmentation = AudioAugmentation(p=0.3)
print("Data augmentation initialized.")

In [ ]:
# ============================================
# SECTION 6: CNN AUDIO CLASSIFIER MODEL
# ============================================

class AudioCNN(nn.Module):
    """CNN for multi-label audio tagging."""
    
    def __init__(self, num_classes, config):
        super().__init__()
        
        self.mel_spec = T.MelSpectrogram(
            sample_rate=config['sample_rate'],
            n_fft=config['n_fft'],
            hop_length=config['hop_length'],
            n_mels=config['n_mels']
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Dropout(0.25),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.mel_spec(x)
        x = self.amplitude_to_db(x)
        x = x.unsqueeze(1)
        x = self.conv_layers(x)
        x = self.classifier(x)
        return x

print("AudioCNN model defined.")

In [ ]:
# ============================================
# SECTION 7: TRANSFORMER GENERATOR MODEL
# ============================================

class AudioTokenizer(nn.Module):
    """Tokenize audio into discrete tokens."""
    
    def __init__(self, codebook_size=1024, embedding_dim=256, max_seq_len=512):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, stride=4, padding=3),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=4, padding=2),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, stride=4, padding=2),
            nn.ReLU(),
            nn.Conv1d(256, embedding_dim, kernel_size=3, stride=4, padding=1),
        )
        self.codebook = nn.Embedding(codebook_size, embedding_dim)
        self.codebook_size = codebook_size
        self.embedding_dim = embedding_dim
    
    def encode(self, x):
        x = x.unsqueeze(1)
        z = self.encoder(x)
        z = z.permute(0, 2, 1)
        
        if z.shape[1] > self.max_seq_len:
            z = z[:, :self.max_seq_len, :]
        
        z_flat = z.reshape(-1, self.embedding_dim)
        distances = torch.cdist(z_flat, self.codebook.weight)
        tokens_flat = distances.argmin(dim=-1)
        tokens = tokens_flat.reshape(z.shape[0], z.shape[1])
        
        return tokens, z
    
    def decode_tokens(self, tokens):
        return self.codebook(tokens)


class AudioTransformer(nn.Module):
    """Transformer for audio generation."""
    
    def __init__(self, config, codebook_size=1024, d_model=256, nhead=8, num_layers=4, max_seq_len=512):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tokenizer = AudioTokenizer(codebook_size, d_model, max_seq_len)
        
        self.register_buffer('pos_encoding', self._create_sinusoidal_pos(max_seq_len, d_model))
        self.condition_embed = nn.Embedding(16, d_model)
        self.token_embed = nn.Embedding(codebook_size, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, codebook_size)
        
        self.d_model = d_model
        self.codebook_size = codebook_size
    
    def _create_sinusoidal_pos(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)
    
    def forward(self, x, condition=None):
        tokens, z = self.tokenizer.encode(x)
        seq_len = tokens.shape[1]
        
        token_embeds = self.token_embed(tokens)
        x = token_embeds + self.pos_encoding[:, :seq_len, :]
        
        if condition is not None:
            cond_embed = self.condition_embed(condition).unsqueeze(1)
            x = x + cond_embed
        
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        output = self.transformer(x, mask=causal_mask)
        logits = self.output_proj(output)
        
        return logits, tokens
    
    @torch.no_grad()
    def generate(self, condition, num_tokens=256, temperature=1.0):
        self.eval()
        device = next(self.parameters()).device
        num_tokens = min(num_tokens, self.max_seq_len)
        
        tokens = torch.randint(0, self.codebook_size, (1, 1), device=device)
        
        for _ in range(num_tokens - 1):
            seq_len = tokens.shape[1]
            embeds = self.token_embed(tokens)
            x = embeds + self.pos_encoding[:, :seq_len, :]
            
            if condition is not None:
                cond_embed = self.condition_embed(condition).unsqueeze(1)
                x = x + cond_embed
            
            causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
            output = self.transformer(x, mask=causal_mask)
            logits = self.output_proj(output[:, -1, :]) / temperature
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            tokens = torch.cat([tokens, next_token], dim=1)
        
        return tokens

print("AudioTransformer model defined.")

In [ ]:
# ============================================
# SECTION 8: AUDIO AUTOENCODER
# ============================================

class AudioDecoder(nn.Module):
    """Decode tokens back to audio waveforms."""
    
    def __init__(self, codebook_size=1024, embedding_dim=256, target_length=110250):
        super().__init__()
        self.target_length = target_length
        self.embedding_dim = embedding_dim
        self.codebook = nn.Embedding(codebook_size, embedding_dim)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(embedding_dim, 256, kernel_size=4, stride=4, padding=0),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.ConvTranspose1d(256, 128, kernel_size=4, stride=4, padding=0),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=4, padding=0),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=4, padding=0),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Conv1d(32, 1, kernel_size=7, padding=3),
            nn.Tanh(),
        )
    
    def forward(self, tokens):
        embeddings = self.codebook(tokens)
        x = embeddings.permute(0, 2, 1)
        waveform = self.decoder(x)
        waveform = waveform.squeeze(1)
        
        if waveform.shape[1] < self.target_length:
            padding = self.target_length - waveform.shape[1]
            waveform = F.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, :self.target_length]
        
        return waveform


class AudioAutoencoder(nn.Module):
    """Complete autoencoder: Audio → Tokens → Audio."""
    
    def __init__(self, config, codebook_size=1024, embedding_dim=256):
        super().__init__()
        self.target_length = config['sample_rate'] * config['duration']
        self.tokenizer = AudioTokenizer(codebook_size, embedding_dim)
        self.decoder = AudioDecoder(codebook_size, embedding_dim, self.target_length)
        self.decoder.codebook = self.tokenizer.codebook
    
    def encode(self, audio):
        tokens, embeddings = self.tokenizer.encode(audio)
        return tokens
    
    def decode(self, tokens):
        return self.decoder(tokens)
    
    def forward(self, audio):
        tokens = self.encode(audio)
        reconstructed = self.decode(tokens)
        return reconstructed, tokens

autoencoder = AudioAutoencoder(CONFIG).to(device)
print(f"AudioAutoencoder initialized")
print(f"  Encoder params: {sum(p.numel() for p in autoencoder.tokenizer.parameters()):,}")
print(f"  Decoder params: {sum(p.numel() for p in autoencoder.decoder.parameters()):,}")

In [ ]:
# ============================================
# SECTION 8.5: AUTOENCODER TRAINING WITH PERCEPTUAL LOSS
# ============================================

class MultiScaleSpectralLoss(nn.Module):
    """Multi-scale spectral loss for better perceptual quality."""
    
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[128, 256, 512]):
        super().__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        
    def forward(self, y_pred, y_true):
        total_loss = 0
        
        for fft_size, hop_size in zip(self.fft_sizes, self.hop_sizes):
            # STFT
            window = torch.hann_window(fft_size, device=y_pred.device)
            
            pred_stft = torch.stft(y_pred, fft_size, hop_size, window=window, return_complex=True)
            true_stft = torch.stft(y_true, fft_size, hop_size, window=window, return_complex=True)
            
            # Magnitude spectral loss
            pred_mag = torch.abs(pred_stft)
            true_mag = torch.abs(true_stft)
            
            # Spectral convergence loss
            sc_loss = torch.norm(true_mag - pred_mag, p='fro') / (torch.norm(true_mag, p='fro') + 1e-8)
            
            # Log magnitude loss
            log_pred = torch.log(pred_mag + 1e-8)
            log_true = torch.log(true_mag + 1e-8)
            log_loss = F.l1_loss(log_pred, log_true)
            
            total_loss += sc_loss + log_loss
        
        return total_loss / len(self.fft_sizes)


class PerceptualAudioLoss(nn.Module):
    """Combined perceptual loss: MSE + Multi-scale Spectral + Phase."""
    
    def __init__(self, mse_weight=0.1, spectral_weight=1.0, phase_weight=0.1):
        super().__init__()
        self.mse_weight = mse_weight
        self.spectral_weight = spectral_weight
        self.phase_weight = phase_weight
        self.spectral_loss = MultiScaleSpectralLoss()
    
    def forward(self, y_pred, y_true):
        # Ensure same length
        min_len = min(y_pred.shape[-1], y_true.shape[-1])
        y_pred = y_pred[..., :min_len]
        y_true = y_true[..., :min_len]
        
        # MSE loss (time domain)
        mse_loss = F.mse_loss(y_pred, y_true)
        
        # Multi-scale spectral loss
        spectral_loss = self.spectral_loss(y_pred, y_true)
        
        # Phase-aware loss (gradient correlation)
        pred_diff = y_pred[..., 1:] - y_pred[..., :-1]
        true_diff = y_true[..., 1:] - y_true[..., :-1]
        phase_loss = F.mse_loss(pred_diff, true_diff)
        
        total = (self.mse_weight * mse_loss + 
                 self.spectral_weight * spectral_loss + 
                 self.phase_weight * phase_loss)
        
        return total, {'mse': mse_loss.item(), 'spectral': spectral_loss.item(), 'phase': phase_loss.item()}


class ImprovedAudioDecoder(nn.Module):
    """Improved decoder with residual connections and more capacity."""
    
    def __init__(self, codebook_size=1024, embedding_dim=256, target_length=110250):
        super().__init__()
        self.target_length = target_length
        self.embedding_dim = embedding_dim
        self.codebook = nn.Embedding(codebook_size, embedding_dim)
        
        # Upsampling with residual blocks
        self.upsample1 = nn.ConvTranspose1d(embedding_dim, 256, kernel_size=8, stride=4, padding=2)
        self.res1 = nn.Sequential(
            nn.Conv1d(256, 256, 3, padding=1), nn.ReLU(),
            nn.Conv1d(256, 256, 3, padding=1)
        )
        
        self.upsample2 = nn.ConvTranspose1d(256, 128, kernel_size=8, stride=4, padding=2)
        self.res2 = nn.Sequential(
            nn.Conv1d(128, 128, 3, padding=1), nn.ReLU(),
            nn.Conv1d(128, 128, 3, padding=1)
        )
        
        self.upsample3 = nn.ConvTranspose1d(128, 64, kernel_size=8, stride=4, padding=2)
        self.res3 = nn.Sequential(
            nn.Conv1d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1)
        )
        
        self.upsample4 = nn.ConvTranspose1d(64, 32, kernel_size=8, stride=4, padding=2)
        self.final = nn.Conv1d(32, 1, kernel_size=7, padding=3)
        self.activation = nn.Tanh()
    
    def forward(self, tokens):
        embeddings = self.codebook(tokens)
        x = embeddings.permute(0, 2, 1)
        
        x = F.relu(self.upsample1(x))
        x = x + self.res1(x)
        
        x = F.relu(self.upsample2(x))
        x = x + self.res2(x)
        
        x = F.relu(self.upsample3(x))
        x = x + self.res3(x)
        
        x = F.relu(self.upsample4(x))
        waveform = self.activation(self.final(x)).squeeze(1)
        
        if waveform.shape[1] < self.target_length:
            padding = self.target_length - waveform.shape[1]
            waveform = F.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, :self.target_length]
        
        return waveform


class ImprovedAudioAutoencoder(nn.Module):
    """Autoencoder with improved decoder and shared codebook."""
    
    def __init__(self, config, codebook_size=1024, embedding_dim=256):
        super().__init__()
        self.target_length = config['sample_rate'] * config['duration']
        self.tokenizer = AudioTokenizer(codebook_size, embedding_dim)
        self.decoder = ImprovedAudioDecoder(codebook_size, embedding_dim, self.target_length)
        self.decoder.codebook = self.tokenizer.codebook
    
    def encode(self, audio):
        tokens, _ = self.tokenizer.encode(audio)
        return tokens
    
    def decode(self, tokens):
        return self.decoder(tokens)
    
    def forward(self, audio):
        tokens = self.encode(audio)
        reconstructed = self.decode(tokens)
        return reconstructed, tokens


def train_autoencoder(autoencoder, train_loader, val_loader, config, epochs=5):
    """Train autoencoder with perceptual loss."""
    autoencoder.to(device)
    criterion = PerceptualAudioLoss(mse_weight=0.1, spectral_weight=1.0, phase_weight=0.1)
    optimizer = torch.optim.AdamW(autoencoder.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'train_loss': [], 'val_loss': [], 'val_snr': []}
    best_snr = -float('inf')
    
    for epoch in range(epochs):
        # Training
        autoencoder.train()
        train_loss = 0
        
        for batch_idx, (audio, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')):
            audio = audio.to(device)
            
            optimizer.zero_grad()
            reconstructed, tokens = autoencoder(audio)
            loss, loss_components = criterion(reconstructed, audio)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(autoencoder.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            
            if batch_idx % 50 == 0:
                print(f"  Batch {batch_idx}: MSE={loss_components['mse']:.4f}, "
                      f"Spectral={loss_components['spectral']:.4f}, Phase={loss_components['phase']:.4f}")
        
        train_loss /= len(train_loader)
        history['train_loss'].append(train_loss)
        
        # Validation
        autoencoder.eval()
        val_loss = 0
        total_snr = 0
        num_val = 0
        
        with torch.no_grad():
            for audio, _ in val_loader:
                audio = audio.to(device)
                reconstructed, _ = autoencoder(audio)
                loss, _ = criterion(reconstructed, audio)
                val_loss += loss.item()
                
                # Calculate SNR
                min_len = min(reconstructed.shape[-1], audio.shape[-1])
                audio_trim = audio[..., :min_len]
                recon_trim = reconstructed[..., :min_len]
                
                noise = audio_trim - recon_trim
                signal_power = (audio_trim ** 2).mean()
                noise_power = (noise ** 2).mean()
                
                if noise_power > 0:
                    snr = 10 * torch.log10(signal_power / noise_power).item()
                    total_snr += snr
                    num_val += 1
        
        val_loss /= len(val_loader)
        avg_snr = total_snr / max(num_val, 1)
        
        history['val_loss'].append(val_loss)
        history['val_snr'].append(avg_snr)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val SNR={avg_snr:.2f}dB")
        
        # Save best model
        if avg_snr > best_snr:
            best_snr = avg_snr
            torch.save(autoencoder.state_dict(), config['model_save_dir'] / 'autoencoder_best.pth')
            print(f"  Saved best model with SNR={avg_snr:.2f}dB")
        
        scheduler.step()
    
    return history


def validate_audio_quality(autoencoder, val_loader, num_samples=5):
    """Validate audio reconstruction quality with detailed metrics."""
    autoencoder.eval()
    results = []
    
    with torch.no_grad():
        for idx, (audio, _) in enumerate(val_loader):
            if idx >= num_samples:
                break
            
            audio = audio.to(device)
            reconstructed, _ = autoencoder(audio)
            
            # Get first sample from batch
            orig = audio[0].cpu().numpy()
            recon = reconstructed[0].cpu().numpy()
            
            min_len = min(len(orig), len(recon))
            orig = orig[:min_len]
            recon = recon[:min_len]
            
            # Calculate metrics
            mse = np.mean((orig - recon) ** 2)
            signal_power = np.mean(orig ** 2)
            noise_power = np.mean((orig - recon) ** 2)
            snr = 10 * np.log10(signal_power / max(noise_power, 1e-10))
            
            # Dynamic range
            orig_range = np.max(np.abs(orig)) - np.min(np.abs(orig))
            recon_range = np.max(np.abs(recon)) - np.min(np.abs(recon))
            dynamic_preservation = min(recon_range / max(orig_range, 1e-10), 1.0)
            
            results.append({
                'sample_idx': idx,
                'mse': mse,
                'snr': snr,
                'orig_range': [float(orig.min()), float(orig.max())],
                'recon_range': [float(recon.min()), float(recon.max())],
                'dynamic_preservation': dynamic_preservation,
            })
            
            print(f"Sample {idx}: SNR={snr:.2f}dB, MSE={mse:.6f}, "
                  f"Orig range=[{orig.min():.4f}, {orig.max():.4f}], "
                  f"Recon range=[{recon.min():.4f}, {recon.max():.4f}]")
    
    avg_snr = np.mean([r['snr'] for r in results])
    avg_mse = np.mean([r['mse'] for r in results])
    avg_dynamic = np.mean([r['dynamic_preservation'] for r in results])
    
    print(f"\nAverage SNR: {avg_snr:.2f}dB")
    print(f"Average MSE: {avg_mse:.6f}")
    print(f"Average Dynamic Preservation: {avg_dynamic:.2%}")
    
    # Quality check
    if avg_snr >= 20:
        print("✅ PASS: Audio quality acceptable (SNR >= 20dB)")
    else:
        print("❌ FAIL: Audio quality below threshold (SNR < 20dB)")
    
    return results

print("Perceptual loss and improved autoencoder defined.")

In [ ]:
# ============================================
# SECTION 9: SVDQuant QUANTIZATION
# ============================================

class SVDQuantAudio:
    """Phase-aware audio model quantization."""
    
    def __init__(self, bit_depth=8):
        self.bit_depth = bit_depth
        self.scale_factors = {}
        
        if bit_depth <= 4:
            self.target_fad = 0.25
            self.transient_threshold = 0.3
        elif bit_depth <= 8:
            self.target_fad = 0.15
            self.transient_threshold = 0.5
        else:
            self.target_fad = 0.05
            self.transient_threshold = 0.7
    
    def detect_transients(self, tensor):
        if tensor.dim() < 2:
            return torch.zeros_like(tensor, dtype=torch.bool)
        
        diff = torch.abs(torch.diff(tensor, dim=-1))
        threshold = diff.mean() + 2 * diff.std()
        transients = diff > threshold
        
        padding = torch.zeros(*tensor.shape[:-1], 1, dtype=torch.bool, device=tensor.device)
        return torch.cat([padding, transients], dim=-1)
    
    def apply_dithering(self, tensor):
        noise1 = torch.rand_like(tensor) - 0.5
        noise2 = torch.rand_like(tensor) - 0.5
        dither = (noise1 + noise2) / (2 ** self.bit_depth)
        return tensor + dither
    
    def quantize_tensor(self, tensor, preserve_transients=True):
        max_val = tensor.abs().max()
        if max_val == 0:
            return tensor, {'scale': 1.0}
        
        scale = max_val / (2 ** (self.bit_depth - 1) - 1)
        dithered = self.apply_dithering(tensor)
        quantized = torch.round(dithered / scale) * scale
        
        if preserve_transients and tensor.dim() >= 2:
            transient_mask = self.detect_transients(tensor)
            if transient_mask.any():
                high_precision_scale = scale * 0.5
                high_precision = torch.round(tensor / high_precision_scale) * high_precision_scale
                quantized = torch.where(transient_mask, high_precision, quantized)
        
        return quantized, {'scale': scale.item()}
    
    def quantize_model(self, model):
        import copy
        quantized_model = copy.deepcopy(model)
        quantized_state = {}
        
        for name, param in model.named_parameters():
            if param.requires_grad:
                quantized, meta = self.quantize_tensor(param.data)
                quantized_state[name] = quantized
                self.scale_factors[name] = meta['scale']
        
        state_dict = quantized_model.state_dict()
        for name, quantized_param in quantized_state.items():
            state_dict[name] = quantized_param
        quantized_model.load_state_dict(state_dict)
        
        return quantized_model
    
    def calculate_compression_ratio(self, model):
        return 32.0 / self.bit_depth

print("SVDQuantAudio quantization class defined.")

In [ ]:
# ============================================
# SECTION 10: TRAINER CLASS
# ============================================

from sklearn.metrics import roc_auc_score

class Trainer:
    """Training orchestrator for all models."""
    
    def __init__(self, model, config, model_type='classifier'):
        self.model = model.to(device)
        self.config = config
        self.model_type = model_type
        
        if model_type == 'classifier':
            self.criterion = nn.BCEWithLogitsLoss()
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=config['epochs'])
        self.history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    
    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0
        
        for data, labels in tqdm(dataloader, desc='Training'):
            data, labels = data.to(device), labels.to(device)
            
            self.optimizer.zero_grad()
            
            if self.model_type == 'classifier':
                outputs = self.model(data)
                loss = self.criterion(outputs, labels)
            else:
                logits, tokens = self.model(data)
                loss = self.criterion(
                    logits[:, :-1].reshape(-1, logits.shape[-1]),
                    tokens[:, 1:].reshape(-1)
                )
            
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        
        return total_loss / len(dataloader)
    
    @torch.no_grad()
    def validate(self, dataloader):
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_labels = []
        
        for data, labels in dataloader:
            data, labels = data.to(device), labels.to(device)
            
            if self.model_type == 'classifier':
                outputs = self.model(data)
                loss = self.criterion(outputs, labels)
                all_preds.append(torch.sigmoid(outputs).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
            else:
                logits, tokens = self.model(data)
                loss = self.criterion(
                    logits[:, :-1].reshape(-1, logits.shape[-1]),
                    tokens[:, 1:].reshape(-1)
                )
            
            total_loss += loss.item()
        
        val_auc = 0.5
        if self.model_type == 'classifier' and all_preds:
            try:
                preds = np.vstack(all_preds)
                labels = np.vstack(all_labels)
                # Filter out columns where labels have only one class (all 0s or all 1s)
                valid_cols = [i for i in range(labels.shape[1]) if len(np.unique(labels[:, i])) > 1]
                if len(valid_cols) > 0:
                    val_auc = roc_auc_score(labels[:, valid_cols], preds[:, valid_cols], average='macro')
            except Exception as e:
                print(f"AUC calculation error: {e}")
        
        return total_loss / len(dataloader), val_auc
    
    def train(self, train_loader, val_loader, epochs):
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss, val_auc = self.validate(val_loader)
            
            self.scheduler.step()
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_auc'].append(val_auc)
            
            print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | AUC: {val_auc:.4f}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                self.save_checkpoint(f"{self.model_type}_best.pth")
        
        return self.history
    
    def save_checkpoint(self, filename):
        path = self.config['model_save_dir'] / filename
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'history': self.history,
        }, path)
        print(f"Model saved to {path}")

print("Trainer class defined.")

In [ ]:
# ============================================
# STEP 1: CREATE DATASETS
# ============================================

print("\n" + "="*60)
print("STEP 1: Creating datasets...")
print("="*60)

train_dataset = MagnaTagATuneDataset(CONFIG, split='train', transform=augmentation)
val_dataset = MagnaTagATuneDataset(CONFIG, split='val')

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

num_classes = len(train_dataset.tag_columns)
print(f"Number of classes: {num_classes}")

In [ ]:
# ============================================
# STEP 2: TRAIN CNN CLASSIFIER
# ============================================

print("\n" + "="*60)
print("STEP 2: Training CNN Classifier...")
print("="*60)

cnn_model = AudioCNN(num_classes, CONFIG)
cnn_trainer = Trainer(cnn_model, CONFIG, model_type='classifier')

cnn_history = cnn_trainer.train(train_loader, val_loader, epochs=CONFIG['epochs'])

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(cnn_history['train_loss'], label='Train')
plt.plot(cnn_history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN Classifier Training')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(cnn_history['val_auc'], 'g-')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.title('Validation AUC')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Best AUC: {max(cnn_history['val_auc']):.4f}")

In [ ]:
# ============================================
# STEP 2.5: TRAIN AUDIO AUTOENCODER (with perceptual loss)
# ============================================

print("\n" + "="*60)
print("STEP 2.5: Training Audio Autoencoder...")
print("="*60)

improved_autoencoder = ImprovedAudioAutoencoder(CONFIG).to(device)
print(f"ImprovedAudioAutoencoder initialized")
print(f"  Total params: {sum(p.numel() for p in improved_autoencoder.parameters()):,}")

autoencoder_history = train_autoencoder(
    improved_autoencoder, 
    train_loader, 
    val_loader, 
    CONFIG, 
    epochs=5
)

# Plot autoencoder training
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(autoencoder_history['train_loss'], label='Train')
plt.plot(autoencoder_history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Perceptual Loss')
plt.title('Autoencoder Training')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(autoencoder_history['val_snr'], 'g-')
plt.xlabel('Epoch')
plt.ylabel('SNR (dB)')
plt.title('Validation SNR')
plt.axhline(y=20, color='r', linestyle='--', label='Quality Threshold')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Validate audio quality
print("\n" + "="*60)
print("Validating Audio Reconstruction Quality...")
print("="*60)
quality_results = validate_audio_quality(improved_autoencoder, val_loader, num_samples=5)

print(f"\nBest SNR achieved: {max(autoencoder_history['val_snr']):.2f}dB")

In [ ]:
# ============================================
# STEP 3: TRAIN TRANSFORMER GENERATOR
# ============================================

print("\n" + "="*60)
print("STEP 3: Training Transformer Generator...")
print("="*60)

transformer_model = AudioTransformer(CONFIG)
transformer_trainer = Trainer(transformer_model, CONFIG, model_type='generator')

transformer_history = transformer_trainer.train(train_loader, val_loader, epochs=5)

plt.figure(figsize=(10, 4))
plt.plot(transformer_history['train_loss'], label='Train')
plt.plot(transformer_history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer Generator Training')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================
# STEP 4: QUANTIZE MODELS
# ============================================

print("\n" + "="*60)
print("STEP 4: Quantizing models with SVDQuant...")
print("="*60)

# Quantize CNN
quantizer_8bit = SVDQuantAudio(bit_depth=8)
quantized_cnn = quantizer_8bit.quantize_model(cnn_model)
cnn_compression = quantizer_8bit.calculate_compression_ratio(quantized_cnn)
print(f"CNN 8-bit compression ratio: {cnn_compression:.2f}x")

# Quantize Transformer
quantizer_4bit = SVDQuantAudio(bit_depth=4)
quantized_transformer = quantizer_4bit.quantize_model(transformer_model)
transformer_compression = quantizer_4bit.calculate_compression_ratio(quantized_transformer)
print(f"Transformer 4-bit compression ratio: {transformer_compression:.2f}x")

# Save quantized models
torch.save(quantized_cnn.state_dict(), CONFIG['model_save_dir'] / 'cnn_quantized_8bit.pth')
torch.save(quantized_transformer.state_dict(), CONFIG['model_save_dir'] / 'transformer_quantized_4bit.pth')
print("\nQuantized models saved!")

In [ ]:
# ============================================
# STEP 5: GENERATE SAMPLE AUDIO
# ============================================

print("\n" + "="*60)
print("STEP 5: Generating sample audio...")
print("="*60)

condition = torch.tensor([0], device=device)  # Amapiano style
generated_tokens = quantized_transformer.generate(condition, num_tokens=256, temperature=0.9)

print(f"Generated {generated_tokens.shape[1]} tokens")
print(f"Token range: {generated_tokens.min().item()} - {generated_tokens.max().item()}")

In [ ]:
# ============================================
# STEP 6: EXPORT TO ONNX
# ============================================

print("\n" + "="*60)
print("STEP 6: Exporting models to ONNX...")
print("="*60)

try:
    dummy_input = torch.randn(1, CONFIG['sample_rate'] * CONFIG['duration']).to(device)
    onnx_path = CONFIG['output_dir'] / 'audio_classifier.onnx'
    
    torch.onnx.export(
        quantized_cnn,
        dummy_input,
        str(onnx_path),
        input_names=['audio'],
        output_names=['tags'],
        dynamic_axes={'audio': {0: 'batch'}, 'tags': {0: 'batch'}},
        opset_version=18
    )
    print(f"✅ CNN exported to ONNX: {onnx_path}")
except Exception as e:
    print(f"⚠️ ONNX export failed: {e}")

# Save config for web deployment
import json
config_export = {
    'sample_rate': CONFIG['sample_rate'],
    'duration': CONFIG['duration'],
    'n_mels': CONFIG['n_mels'],
    'n_fft': CONFIG['n_fft'],
    'hop_length': CONFIG['hop_length'],
    'target_tags': CONFIG['target_tags'],
}

with open(CONFIG['output_dir'] / 'model_config.json', 'w') as f:
    json.dump(config_export, f, indent=2)

print(f"✅ Config saved to {CONFIG['output_dir'] / 'model_config.json'}")

In [ ]:
# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "="*60)
print("✅ TRAINING PIPELINE COMPLETE")
print("="*60)
print(f"\nModels saved to: {CONFIG['model_save_dir']}")
print("\nGenerated files:")
for f in CONFIG['model_save_dir'].glob('*'):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name}: {size_mb:.2f} MB")

print("\n📊 Training Summary:")
print(f"  CNN Best AUC: {max(cnn_history['val_auc']):.4f}")
print(f"  CNN Final Train Loss: {cnn_history['train_loss'][-1]:.4f}")
print(f"  CNN Final Val Loss: {cnn_history['val_loss'][-1]:.4f}")
print(f"  Autoencoder Best SNR: {max(autoencoder_history['val_snr']):.2f}dB")
print(f"  Autoencoder Final Val Loss: {autoencoder_history['val_loss'][-1]:.4f}")
print(f"  Transformer Final Train Loss: {transformer_history['train_loss'][-1]:.4f}")
print(f"  Transformer Final Val Loss: {transformer_history['val_loss'][-1]:.4f}")

# Audio quality assessment
best_snr = max(autoencoder_history['val_snr'])
if best_snr >= 20:
    print(f"\n✅ Audio quality PASSED (SNR {best_snr:.2f}dB >= 20dB threshold)")
else:
    print(f"\n⚠️ Audio quality BELOW threshold (SNR {best_snr:.2f}dB < 20dB)")
    print("  Consider: more training epochs, larger model, or different loss weights")

print("\n🚀 Next steps:")
print("  1. Increase epochs for better results")
print("  2. Fine-tune on Amapiano-specific dataset")
print("  3. Deploy ONNX model to web application")
print("  4. Run user study to validate authenticity")
print("  5. Save A/B audio pairs for listening tests")